# Phase 3 — Création de la cible « canular »

Objectifs :
- Recharger les lignes valides du fichier.
- Refaire les conversions de types nécessaires.
- Créer une variable cible artificielle `is_hoax`.
- Compter les relevés étiquetés comme canulars.
- Examiner des exemples.
- Identifier les limites de la règle.

## Imports

In [1]:
from pathlib import Path
import csv
import re
import pandas as pd

## Chemins et colonnes

In [2]:
DATA_PATH = Path("../data/releves_klaxo3.csv")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = [
    "datetime",
    "city",
    "state",
    "country",
    "shape",
    "duration_seconds",
    "duration_hours_min",
    "comments",
    "date_posted",
    "latitude",
    "longitude",
]

## Recharger les lignes valides

In [3]:
lignes_valides = []
lignes_problemes = []

with open(DATA_PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)

    for numero_ligne, row in enumerate(reader, start=1):
        if len(row) == len(COLUMNS):
            lignes_valides.append(row)
        else:
            lignes_problemes.append({
                "numero_ligne": numero_ligne,
                "nb_champs": len(row),
                "contenu": row
            })

df = pd.DataFrame(lignes_valides, columns=COLUMNS)

print(f"Nombre de lignes exploitables : {len(df)}")
print(f"Nombre de lignes structurées à part : {len(lignes_problemes)}")

Nombre de lignes exploitables : 88679
Nombre de lignes structurées à part : 196


## Refaire les conversions utiles

In [4]:
colonnes_numeriques = [
    "duration_seconds",
    "latitude",
    "longitude",
]

colonnes_dates = [
    "datetime",
    "date_posted",
]

for col in colonnes_numeriques:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in colonnes_dates:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df.dtypes

datetime              datetime64[ns]
city                          object
state                         object
country                       object
shape                         object
duration_seconds             float64
duration_hours_min            object
comments                      object
date_posted           datetime64[ns]
latitude                     float64
longitude                    float64
dtype: object

## Afficher quelques commentaires

In [7]:
pd.set_option("display.max_colwidth", None)

df[
    [
        "datetime",
        "city",
        "country",
        "shape",
        "comments"
    ]
].sample(
    n=10,
    random_state=42
)

,datetime,city,country,shape,comments
48205,2004-05-21 22:45:00,st-philippe (canada),,triangle,HBCCUFO CANADIAN REPORT: triangle
73977,1975-08-16 20:00:00,highlands,us,fireball,Ball of light like a roman candle
66019,2013-07-24 00:00:00,tulsa,us,light,Green beam of light over Tulsa
38590,2009-03-05 20:05:00,west valley city,,circle,Objects hovering near and above Salt Lake City and other one flying in zig zag pattern
29694,2013-02-15 19:00:00,arlington heights,us,fireball,We saw 4 big bright orange fire ball type flying slowly and disappearing 1 by 1.
5719,2012-10-28 16:00:00,bourbonnais,us,circle,Metallic blue round object over bourbonnais&#44 IL dashes out of sight in 3 seconds
12066,1998-01-01 20:00:00,graniteville,us,disk,I was taking pictures of a full moon and did not see it until the pictures were developed. I don&#39t know what to do with the picture (it
618,1985-10-01 05:30:00,monroe,us,sphere,Years ago&#44 my husband&#39s aunt and I witnessed something very strange on the way to work early one morning. It was about 5:30am and we we
78087,2005-08-30 21:30:00,kentfield,us,disk,UFO exiting our atmosphere.
61290,1980-07-01 22:00:00,netherlands,,unknown,Big black object with coloured lights


## Afficher les mots liés à des canulars

In [8]:
df.loc[
    df["comments"].fillna("").str.contains(
        "hoax|fake|prank|joke",
        case=False,
        regex=True
    ),
    [
        "datetime",
        "city",
        "country",
        "comments"
    ]
].head(20)

,datetime,city,country,comments
658,1994-10-01 13:13:00,new york city,us,a flying colorful disc above my car&#44 near Erie. ((NUFORC Note: Possible hoax?? PD))
808,2004-10-01 17:00:00,las vegas,us,((HOAX??)) Short encounter with space craft on my way into my parking lot area.
958,2008-10-01 19:12:00,bonham,us,Silver egg shape over six houses. ((NUFORC Note: Possible hoax?? PD))
1207,2007-10-12 22:00:00,irvine,us,Lights in Irvine October 2007: Hoax
1211,2007-10-12 23:00:00,rogers,us,((HOAX??)) abduction. 500 Lights On Object0: Yes
1492,2009-10-13 06:30:00,troy,us,((HOAX??)) Flying craft which was big as a football field
1582,2013-10-13 10:50:00,santa fe,us,((HOAX??)) Some kind of aircraft with a HUGE wingspan was flying very low over my neighborhood in Santa Fe&#44 NM.
1727,2006-10-14 02:00:00,yuma,us,((HOAX??)) two aliens appeared from a bright light to peacefully investigate the surroundings in the woods
1740,2007-10-14 05:00:00,rawalpindi (pakistan),,((HOAX)) usaually i stand near the airport. on that day i saw that there were
1965,NaT,greenwich,us,((HOAX??)) had no control on tv. ash tray killd itself. the obkect was one big light.


## Définir la règle

### Règle retenue

Un relevé est étiqueté comme canular lorsque son commentaire contient au moins
un mot-clé explicitement associé à une fraude, une mise en scène ou une
plaisanterie.